# ML Testing & MLOps: Reproducibility and Serialization Contract Tests

**Dataset:** [Wine recognition dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#wine-recognition-dataset) (`sklearn.datasets.load_wine`) — 178 real chemical analyses of
wines grown in the same region of Italy, from three different cultivars, with 13 real-valued
features (alcohol, malic acid, ash, flavanoids, color intensity, proline, etc.).

**Problem statement:** A model that trains "successfully" in a notebook is not the same thing as a
model that is *safe to ship*. In production ML systems, the quality-engineering job is to catch the
failure modes that unit tests for ordinary software don't cover: a retrain that silently produces a
different model than last time, a serialized artifact that doesn't reproduce the in-memory model's
predictions, or a serving pipeline that receives columns in the wrong order and silently mispredicts
instead of crashing. You are going to build a small **test suite** (plain `assert`-based functions,
no `pytest` required) that guards a `RandomForestClassifier` trained on the Wine dataset against
exactly these failure modes, and then you are going to deliberately break two of the guarantees to
confirm your tests actually catch the breakage.

**What you should produce**, as working code in this notebook:
1. A `train_model(random_state)` function that loads the Wine data, splits it, and returns a
   fitted `RandomForestClassifier` together with the held-out test split.
2. A `FeatureContractError` exception and a `validate_feature_contract(df, expected_columns)`
   function that raises it if a DataFrame's column names, order, or count don't match what the
   model was trained on.
3. Four test functions, each returning `True`/raising `AssertionError` on failure:
   - `test_reproducible_training` — training twice with the same `random_state` yields identical
     predictions on the test set.
   - `test_nondeterministic_training_is_caught` — training twice **without** fixing
     `random_state` on a model configuration that is sensitive to it should *fail* the "identical
     predictions" check (i.e. your test suite must correctly flag the absence of a seed as a bug).
   - `test_serialization_roundtrip` — a model serialized with `joblib` and reloaded produces
     bit-identical predictions and probabilities to the in-memory model.
   - `test_feature_contract_enforced` — a deployment-shaped input with a shuffled/renamed column
     is rejected by `validate_feature_contract` instead of being silently fed to the model.
4. A short markdown summary (2-4 sentences) of which of these four checks would have caught a real
   incident you can imagine happening in a careless ML deployment, and why.


## Starter cells

Fill in the `TODO`s. Don't look at the solution until you've tried each function yourself.

In [ ]:
import hashlib
import io

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
FEATURE_COLUMNS = list(X.columns)

print(X.shape, y.shape)
X.head()


In [ ]:
class FeatureContractError(Exception):
    """Raised when an input DataFrame doesn't match the schema a model was trained on."""


def validate_feature_contract(df: pd.DataFrame, expected_columns: list[str]) -> None:
    # TODO: raise FeatureContractError if df.columns (as a list, in order) != expected_columns.
    # Make the error message name exactly what's wrong: missing columns, extra columns,
    # or a reordering.
    raise NotImplementedError


In [ ]:
def train_model(random_state, n_estimators=50):
    # TODO:
    #  1. split X, y into train/test with test_size=0.3 and the given random_state
    #  2. fit a RandomForestClassifier(n_estimators=n_estimators, random_state=random_state)
    #     on the training split
    #  3. return (model, X_test, y_test)
    raise NotImplementedError


In [ ]:
def test_reproducible_training():
    # TODO: call train_model(...) twice with the SAME random_state and n_estimators,
    # predict on each model's own X_test, and assert the two prediction arrays are identical
    # (note the two calls may get different X_test splits only if random_state differs --
    # keep it fixed here).
    raise NotImplementedError


def test_nondeterministic_training_is_caught():
    # TODO: call train_model(...) twice with DIFFERENT random_state values (simulating a
    # pipeline that forgot to pin a seed), predict on a SHARED held-out set built once outside
    # train_model, and assert that the predictions are NOT guaranteed identical -- i.e. this
    # test should demonstrate that your "identical predictions" check is actually sensitive
    # to the seed, by showing at least one mismatch across many bootstrapped predictions,
    # or by asserting the two models' feature_importances_ differ.
    raise NotImplementedError


def test_serialization_roundtrip():
    # TODO: train a model, serialize it with joblib.dump to an in-memory buffer (io.BytesIO),
    # reload it with joblib.load, and assert predict() and predict_proba() are identical
    # between the original and reloaded model on X_test.
    raise NotImplementedError


def test_feature_contract_enforced():
    # TODO: build a copy of X_test with two columns swapped (or one renamed), and assert that
    # validate_feature_contract raises FeatureContractError for it, while the unmodified
    # X_test passes with no error.
    raise NotImplementedError


In [ ]:
# TODO: run all four tests and print PASS/FAIL for each.
for test_fn in [test_reproducible_training, test_nondeterministic_training_is_caught,
                test_serialization_roundtrip, test_feature_contract_enforced]:
    raise NotImplementedError


_Your 2-4 sentence summary here: which check would have caught a real incident, and why?_

TODO


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

```python
import hashlib
import io

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
FEATURE_COLUMNS = list(X.columns)


class FeatureContractError(Exception):
    """Raised when an input DataFrame doesn't match the schema a model was trained on."""


def validate_feature_contract(df: pd.DataFrame, expected_columns: list[str]) -> None:
    actual = list(df.columns)
    expected = list(expected_columns)
    if actual == expected:
        return
    missing = [c for c in expected if c not in actual]
    extra = [c for c in actual if c not in expected]
    if missing or extra:
        raise FeatureContractError(
            f"schema mismatch: missing={missing}, extra={extra}"
        )
    # same set of columns, different order
    raise FeatureContractError(
        f"column order mismatch: expected {expected}, got {actual}"
    )


def train_model(random_state, n_estimators=50):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=random_state, stratify=y
    )
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state)
    model.fit(X_train, y_train)
    return model, X_test, y_test


def test_reproducible_training():
    model_a, X_test_a, _ = train_model(random_state=42)
    model_b, X_test_b, _ = train_model(random_state=42)
    preds_a = model_a.predict(X_test_a)
    preds_b = model_b.predict(X_test_b)
    assert np.array_equal(preds_a, preds_b), "same seed produced different predictions"
    assert np.array_equal(
        model_a.feature_importances_, model_b.feature_importances_
    ), "same seed produced different feature importances"
    return True


def test_nondeterministic_training_is_caught():
    # Build one shared held-out set so both models are scored on the same inputs.
    X_train, X_shared_test, y_train, _ = train_test_split(
        X, y, test_size=0.3, random_state=0, stratify=y
    )
    model_a = RandomForestClassifier(n_estimators=10, random_state=1).fit(X_train, y_train)
    model_b = RandomForestClassifier(n_estimators=10, random_state=2).fit(X_train, y_train)
    preds_a = model_a.predict(X_shared_test)
    preds_b = model_b.predict(X_shared_test)
    # With different seeds, a small forest's predictions are not guaranteed identical.
    # Assert the "identical predictions" check would actually fire here.
    assert not np.array_equal(preds_a, preds_b), (
        "expected different seeds to produce at least one differing prediction, "
        "but the suite's identical-predictions check would have passed silently"
    )
    return True


def test_serialization_roundtrip():
    model, X_test, _ = train_model(random_state=42)
    buffer = io.BytesIO()
    joblib.dump(model, buffer)
    buffer.seek(0)
    reloaded = joblib.load(buffer)

    assert np.array_equal(model.predict(X_test), reloaded.predict(X_test)), (
        "predictions changed after serialization round-trip"
    )
    assert np.allclose(
        model.predict_proba(X_test), reloaded.predict_proba(X_test)
    ), "predicted probabilities changed after serialization round-trip"
    return True


def test_feature_contract_enforced():
    _, X_test, _ = train_model(random_state=42)

    # Unmodified input must pass.
    validate_feature_contract(X_test, FEATURE_COLUMNS)

    # Swap two columns' positions -- same columns, wrong order, a classic
    # "serving code zipped the wrong list" bug.
    shuffled = X_test.copy()
    cols = list(shuffled.columns)
    cols[0], cols[1] = cols[1], cols[0]
    shuffled = shuffled[cols]

    try:
        validate_feature_contract(shuffled, FEATURE_COLUMNS)
        raised = False
    except FeatureContractError:
        raised = True
    assert raised, "contract check did not catch a reordered-column input"
    return True


for test_fn in [
    test_reproducible_training,
    test_nondeterministic_training_is_caught,
    test_serialization_roundtrip,
    test_feature_contract_enforced,
]:
    try:
        test_fn()
        print(f"PASS: {test_fn.__name__}")
    except AssertionError as e:
        print(f"FAIL: {test_fn.__name__}: {e}")
```

**Why each check matters:**

- `test_reproducible_training` catches the classic "I retrained and now I get a different model"
  incident — if `random_state` isn't pinned (or an upstream library's defaults change), two
  "identical" training runs can silently diverge, breaking any downstream comparison, A/B test, or
  rollback plan that assumes determinism.
- `test_nondeterministic_training_is_caught` is a **test of the test** — a sanity check that your
  reproducibility assertion is actually discriminating (it would fire on real divergence) rather
  than vacuously passing because, say, you compared a model against itself. SQA instincts call this
  "testing your test oracle," and it is just as important for ML checks as it is for ordinary
  assertions.
- `test_serialization_roundtrip` catches the incident where a model trained in one environment
  (library version, OS, BLAS backend) is pickled, shipped to a serving container, and produces
  subtly different predictions after unpickling — a bug that is invisible until it's already in
  production, because the model still "loads fine."
- `test_feature_contract_enforced` catches **training/serving skew**: a feature-engineering step
  upstream of serving reorders or renames a column, and instead of raising a clear error, a
  position-sensitive model silently scores the wrong feature as the wrong thing and returns a
  confident, wrong prediction with no stack trace to follow.

</details>
